# 04 - Collapse prediction model (SPEC section 8.1)
Gradient-boosted classifiers for P(collapse within 12 months) and P(collapse within 24 months), trained on `gold_entity_features`. Labels come from real ASIC register status changes (REGD to EXAD/DRGD). Each model is tracked in MLflow, registered in Unity Catalog as `<catalog>.<schema>.collapse_model_<h>m`, and given the alias `champion` when it beats the minimum AUC.

Scores are **calibrated**: Platt scaling on held-out, later data, re-weighted from the slice's deliberately over-sampled collapse share (35%) to the collapse rate observed on the full public register (P1). Calibration parameters are versioned in `collapse_calibration` and as model-version tags; scoring (05) applies them.


In [0]:
%pip install -q shap scikit-learn mlflow
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("random_seed", "42")
dbutils.widgets.text("min_auc", "0.65")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
SEED, MIN_AUC = int(dbutils.widgets.get("random_seed")), float(dbutils.widgets.get("min_auc"))
assert CATALOG and SCHEMA
FQ = f"{CATALOG}.{SCHEMA}"

FEATURES = ["company_age_years", "is_licensee", "ato_debt_latest_aud", "ato_debt_trend_6m_pct", "lodgement_on_time_pct",
            "stp_headcount_trend_pct", "gst_cancelled", "smr_count_24m", "large_flow_smr_count_24m", "apra_flag_count",
            "afca_complaints_latest", "afca_complaints_trend_pct", "breach_reports_12m", "enforcement_alternatives_count",
            "banned_director_count", "director_prior_failed_companies", "related_collapsed_entities"]

In [ ]:
from pyspark.sql import functions as F
import mlflow, numpy as np, pandas as pd, shap
from mlflow.models import infer_signature
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/{SCHEMA}_collapse_model")
feat = spark.table(f"{FQ}.gold_entity_features").toPandas()

# Observed population collapse rate (SPEC 8.1: insolvent or deregistered), measured on the full public register (P1) for the
# same company types the slice is drawn from. The slice deliberately over-samples collapsed companies (35%), so raw model
# outputs reflect the sample, not the population; calibration re-weights to this observed rate. P1 only carries the last
# ~12 months of deregistrations, so the 24-month rate assumes a constant annual hazard. EXAD has no entry date in P1 and
# is counted as a current-year entry (a small overstatement).
pop = spark.sql(f"""
  WITH c AS (SELECT status, dereg_date, max(dereg_date) OVER () AS last_dereg FROM {FQ}.silver_company
             WHERE type IN ('APTY', 'APUB') AND reg_date >= '2000-01-01')
  SELECT sum(CASE WHEN status = 'DRGD' AND dereg_date > date_sub(last_dereg, 365) THEN 1 ELSE 0 END) AS dereg_12m,
         sum(CASE WHEN status = 'EXAD' THEN 1 ELSE 0 END) AS exad,
         sum(CASE WHEN status <> 'DRGD' OR dereg_date > date_sub(last_dereg, 365) THEN 1 ELSE 0 END) AS at_risk
  FROM c""").first()
POP_RATE = {12: (pop.dereg_12m + pop.exad) / pop.at_risk}
POP_RATE[24] = 1 - (1 - POP_RATE[12]) ** 2
print("population collapse rate (P1):", {h: round(r, 4) for h, r in POP_RATE.items()}, dict(pop.asDict()))

def ece(y, p, w, bins=10):
    """Weighted expected calibration error over equal-count bins."""
    order = np.argsort(p); y, p, w = y[order], p[order], w[order]
    return float(sum(abs(np.average(yb, weights=wb) - np.average(pb, weights=wb)) * wb.sum()
                     for yb, pb, wb in zip(np.array_split(y, bins), np.array_split(p, bins), np.array_split(w, bins))) / w.sum())

results, calibration = {}, []
for h in (12, 24):
    df = feat[feat[f"label_collapse_{h}m"].notna()]
    assert df[f"label_collapse_{h}m"].nunique() == 2, f"{h}m label has a single class"
    # Out-of-time split: train on earlier as-of dates, hold out the latest ~25% of dates and their entities entirely so
    # overlapping entity-quarters cannot leak. Negatives only exist once the full horizon has elapsed, so restrict to dates
    # where both outcomes are observable. The held-out window is split in time again: the first half fits the calibration,
    # the second half evaluates it.
    last_full = df.loc[df[f"label_collapse_{h}m"] == 0, "as_of_date"].max()
    df = df[df["as_of_date"] <= last_full]
    X = df[FEATURES].astype(float).fillna(0.0); y = df[f"label_collapse_{h}m"].astype(int)
    dates = np.sort(df["as_of_date"].unique())
    cutoff, cal_to = dates[int(len(dates) * 0.75)], dates[int(len(dates) * 0.875)]
    test_ent = set(df.loc[df["as_of_date"] >= cutoff, "entity_id"])
    tr = (df["as_of_date"] < cutoff) & ~df["entity_id"].isin(test_ent)
    ca = (df["as_of_date"] >= cutoff) & (df["as_of_date"] < cal_to)
    ev = df["as_of_date"] >= cal_to
    assert all(y[m].nunique() == 2 for m in (tr, ca, ev)), f"{h}m temporal split produced a single class"
    with mlflow.start_run(run_name=f"collapse_{h}m") as run:
        model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, random_state=SEED).fit(X[tr], y[tr])
        # Platt scaling on the model's log-odds, with class weights that move the sample's collapse share to the population rate
        pi = POP_RATE[h]
        def weights(lab):
            s = lab.mean()
            return np.where(lab == 1, pi / s, (1 - pi) / (1 - s))
        cal = LogisticRegression(C=1e6).fit(model.decision_function(X[ca]).reshape(-1, 1), y[ca], sample_weight=weights(y[ca]))
        a, b = float(cal.coef_[0][0]), float(cal.intercept_[0])
        raw = model.predict_proba(X[ev])[:, 1]
        p = 1 / (1 + np.exp(-(a * model.decision_function(X[ev]) + b)))
        w_ev, y_ev = weights(y[ev]), y[ev].to_numpy()
        auc, ap = roc_auc_score(y_ev, p), average_precision_score(y_ev, p)
        metrics = {"test_auc": auc, "test_avg_precision": ap, "sample_base_rate": float(y.mean()), "population_rate": pi,
                   "ece_raw": ece(y_ev, raw, w_ev), "ece_calibrated": ece(y_ev, p, w_ev),
                   "brier_raw": brier_score_loss(y_ev, raw, sample_weight=w_ev), "brier_calibrated": brier_score_loss(y_ev, p, sample_weight=w_ev),
                   "mean_pred_calibrated": float(np.average(p, weights=w_ev))}
        mlflow.log_params({"horizon_months": h, "n_train": int(tr.sum()), "n_calibration": int(ca.sum()), "n_test": int(ev.sum()),
                           "split": "out_of_time", "test_from_as_of": str(cutoff)[:10], "evaluate_from_as_of": str(cal_to)[:10],
                           "calibration": "platt_on_logodds_prior_weighted", "calib_a": a, "calib_b": b, "features": ",".join(FEATURES)})
        mlflow.log_metrics(metrics)
        info = mlflow.sklearn.log_model(model, name="model", input_example=X[tr].head(5),
                                        signature=infer_signature(X[tr], model.predict_proba(X[tr])[:, 1]),
                                        registered_model_name=f"{FQ}.collapse_model_{h}m")
        imp = pd.Series(np.abs(np.asarray(shap.TreeExplainer(model).shap_values(X[ev].head(500)))).mean(0), index=FEATURES)
        mlflow.log_dict(imp.sort_values(ascending=False).to_dict(), "shap_mean_abs.json")
    version = int(info.registered_model_version)
    for k, v in {"calib_a": a, "calib_b": b, "population_rate": pi}.items():
        client.set_model_version_tag(f"{FQ}.collapse_model_{h}m", str(version), k, f"{v:.6f}")
    calibration.append({"horizon_months": h, "model_version": version, "calib_a": a, "calib_b": b, "population_rate": pi,
                        "sample_base_rate": float(y[ca].mean()), "n_calibration": int(ca.sum()), "n_evaluation": int(ev.sum()),
                        "calibrated_from": str(cutoff)[:10], "evaluated_from": str(cal_to)[:10],
                        **{k: float(metrics[k]) for k in ("test_auc", "ece_raw", "ece_calibrated", "brier_raw", "brier_calibrated")}})
    results[h] = (auc, ap)
    print(f"{h}m v{version}  AUC={auc:.3f}  AP={ap:.3f}  ECE raw={metrics['ece_raw']:.3f} -> calibrated={metrics['ece_calibrated']:.3f}"
          f"  mean pred={metrics['mean_pred_calibrated']:.3f} vs population {pi:.3f}  (a={a:.3f}, b={b:.3f})")

# Versioned calibration parameters, read by scoring (05) for the champion version of each horizon
(spark.createDataFrame(pd.DataFrame(calibration)).withColumn("created_at", F.current_timestamp())
      .write.mode("append").option("mergeSchema", True).saveAsTable(f"{FQ}.collapse_calibration"))


In [0]:
client = mlflow.MlflowClient()
for h, (auc, _) in results.items():
    name = f"{FQ}.collapse_model_{h}m"
    latest = max(client.search_model_versions(f"name = '{name}'"), key=lambda v: int(v.version))
    if auc >= MIN_AUC:
        client.set_registered_model_alias(name, "champion", latest.version)
        print(f"{name} v{latest.version} -> champion")
    else:
        print(f"{name} v{latest.version} AUC {auc:.3f} < {MIN_AUC}; champion unchanged")
for h in (12, 24):
    client.get_model_version_by_alias(f"{FQ}.collapse_model_{h}m", "champion")  # fails if no champion exists